# House 2026: a nonpartisan ML walkthrough (notebook)

The same pipeline as `ml/build.py`, one commented step at a time. Each step names the matching chapter/section of
Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed. (section titles only).

**Goal:** predict the winner of every U.S. House district from public, pre-election information; evaluate on an election the
model never saw; then freeze a 2026 forecast and score it after the results.

**Neutrality rule:** every feature is signed (+ Democratic, − Republican). Swapping the parties flips every input.

**Data (all public):**
| Source | What | Access |
|---|---|---|
| FiveThirtyEight `election-results` (CC BY 4.0) | candidate-level House returns 1998–2024 | `raw.githubusercontent.com/fivethirtyeight/election-results/main/election_results_house.csv` |
| Wikipedia House election pages (CC BY-SA 4.0) | per-district Cook PVI, incumbents, candidates, winners, 2026 polling averages | `en.wikipedia.org/w/index.php?title=<year>_United_States_House_of_Representatives_elections&action=raw` |
| Cook Political Report | Cook PVI (reprinted in the Wikipedia tables) | cookpolitical.com |
| FiftyPlusOne (optional) | every 2026 generic-ballot poll | `fiftyplusone.news/api/csv?api_key=KEY&filename=generic.csv` |

## 0 · Setup
Run from the repo root or from `notebooks/`. `ml/fetch_data.py` must have been run once (it writes `data/raw/`).

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "ml"))
import build as B                      # reuse the exact functions the site was built with
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, brier_score_loss
pd.set_option("display.width", 140)

## 1 · Frame the problem  (HOML Ch. 2: *Frame the Problem*, *Select a Performance Measure*)
- **Unit:** one district in one election. **Label:** did the Democrat win (1) or the Republican (0).
- **Inputs known before Election Day:** Cook PVI, national environment, incumbent running, party holding the seat.
- **Metric:** accuracy, but also accuracy on *competitive* seats, Brier score and calibration.
- The *national* question ("who wins the House?") has only 14 data points 1998–2024: a small-data problem (section 4).

## 2 · Get the data  (HOML Ch. 2: *Get the Data*, *Take a Quick Look at the Data Structure*)

In [2]:
res = B.load_results()                         # 538 returns -> two-party votes per district-year
print("fusion rows in naive join that looked third-party:", B.load_results.raw_winner.groupby(["cycle","state_abbrev","office_seat_name"]).rawParty.first().eq("O").sum())
d = pd.concat([B.load_districts(y).merge(res[res.year == y][["district","margin","winner538"]], on="district", how="left")
               for y in B.YEARS], ignore_index=True)
d = d[d.winner.isin(["D","R"])].reset_index(drop=True)
d["y"] = (d.winner == "D").astype(int)
d["contested"] = ((d.has_d == 1) & (d.has_r == 1)).astype(int)
print(d.shape); d.sample(6, random_state=1)

fusion rows in naive join that looked third-party: 202
(2600, 17)


,district,state,pvi,inc_party,inc_running,inc_count,new_seat,has_d,has_r,n_cands,winner,inc_status,year,margin,winner538,y,contested
304,OH-3,OH,17.0,D,1,1,0,1,1,2,D,Incumbent re-elected.,2014,28.129625,D,1,1
2235,CA-52,CA,18.0,D,1,1,0,1,1,2,D,Incumbent re-elected.,2024,32.618455,D,1,1
1401,FL-22,FL,6.0,D,1,1,0,1,1,2,D,Incumbent re-elected.,2020,17.203101,D,1,1
660,MS-1,MS,-16.0,R,1,1,0,1,1,4,R,Incumbent re-elected.,2016,-42.185660,R,0,1
2217,CA-36,CA,21.0,D,1,1,0,1,1,2,D,Incumbent re-elected.,2024,37.436276,D,1,1
836,VA-2,VA,-4.0,R,0,1,0,1,1,2,R,| Incumbent retired. Republican hold.,2016,-22.920801,R,0,1


**Cleaning check:** two independent sources should agree on every winner. (They did not until we fixed fusion voting in NY/CT; see `load_results`.)

In [3]:
both = d.dropna(subset=["winner538"])
print(f"agree: {(both.winner == both.winner538).mean():.2%} of {len(both)}")

agree: 100.00% of 2564


## 3 · Explore  (HOML Ch. 2: *Explore and Visualize the Data*, *Look for Correlations*)

In [4]:
nat = B.national(res)
env = dict(zip(nat.year, nat.env))
c = d[(d.contested == 1) & d.margin.notna()].copy()
c["expected"] = 2 * c.pvi + c.year.map(env)          # rule of thumb: margin ≈ 2×PVI + national margin
print("corr(PVI, margin) =", round(c[["pvi","margin"]].corr().iloc[0,1], 3))
print("mean |margin - expected| =", round((c.margin - c.expected).abs().mean(), 1), "points")
c.groupby(B.features(c, 0).inc).apply(lambda t: (t.margin - t.expected).mean()).rename({-1:"R incumbent",0:"open",1:"D incumbent"})

corr(PVI, margin) = 0.958
mean |margin - expected| = 6.7 points


inc
R incumbent   -3.729322
open           1.363441
D incumbent    4.252123
dtype: float64

## 4 · The small-data national problem  (HOML Ch. 4 *Linear Regression*; Ch. 7 *Random Forests*)

In [5]:
X, y = nat[["env"]].values, nat.seatShare.values
lin, rf = np.zeros(len(y)), np.zeros(len(y))
for tr, te in LeaveOneOut().split(X):
    lin[te] = LinearRegression().fit(X[tr], y[tr]).predict(X[te])
    rf[te] = RandomForestRegressor(200, random_state=42).fit(X[tr], y[tr]).predict(X[te])
print("LOO mean abs error, seats:  linear", round(np.abs(lin-y).mean()*435,1), " forest", round(np.abs(rf-y).mean()*435,1))

LOO mean abs error, seats:  linear 7.9  forest 11.6


## 5 · Features  (HOML Ch. 2: *Prepare the Data*)
All four are signed + Democratic / − Republican.

In [6]:
X_all = B.features(d, d.year.map(env))
X_all.describe().round(2)

,pvi,env,inc,held
count,2600.00,2600.00,2600.00,2600.00
mean,0.37,0.30,-0.02,-0.04
std,16.10,4.74,0.93,0.99
min,-33.00,-5.33,-1.00,-1.00
25%,-12.00,-2.63,-1.00,-1.00
50%,-2.00,-0.61,0.00,-1.00
75%,12.00,3.49,1.00,1.00
max,44.00,9.07,1.00,1.00


## 6 · Split by time + baselines  (HOML Ch. 2 *Create a Test Set*; Ch. 3 *Measuring Accuracy*)
Train 2014–2020, validate 2022, test 2024. A random split would leak each year's national mood.

In [7]:
con = d.contested == 1
tr, va, te = d.year.isin([2014,2016,2018,2020]) & con, (d.year == 2022) & con, (d.year == 2024) & con
yte = d.y[te]
print("PVI-sign baseline on 2024:", round(accuracy_score(yte, X_all.pvi[te] > 0), 4))
print("holder-keeps baseline:   ", round(accuracy_score(yte, X_all.held[te] > 0), 4))

PVI-sign baseline on 2024: 0.9658
holder-keeps baseline:    0.9487


## 7 · Models  (HOML Ch. 4 *Logistic Regression*; Ch. 6 *Decision Trees*; Ch. 7 *Boosting*)

In [8]:
trva = tr | va
models = {"logreg": B.LogReg(), "tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42), "gbm": B.make_gbm()}
comp = (X_all.pvi[te].abs() <= 5).values
for k, m in models.items():
    m.fit(X_all[trva], d.y[trva])
    p = m.predict_proba(X_all[te])[:, 1]
    print(f"{k:7s} acc {accuracy_score(yte, p>=.5):.3f}  competitive {accuracy_score(yte[comp], p[comp]>=.5):.3f} (n={comp.sum()})  Brier {brier_score_loss(yte, p):.4f}")

logreg  acc 0.963  competitive 0.850 (n=80)  Brier 0.0243
tree    acc 0.963  competitive 0.850 (n=80)  Brier 0.0258


gbm     acc 0.966  competitive 0.863 (n=80)  Brier 0.0224


## 8 · Evaluate honestly: leave one election out + neutrality check  (HOML Ch. 3 *Cross-Validation*)

In [9]:
for yr in B.YEARS:
    a, b = (d.year != yr) & con, (d.year == yr) & con
    m = B.LogReg().fit(X_all[a], d.y[a])
    print(yr, f"env {env[yr]:+.1f}", "acc", round(m.score(X_all[b], d.y[b]), 4))
m = models["logreg"]
gap = np.abs(m.predict_proba(X_all[te])[:,1] + m.predict_proba(-X_all[te])[:,1] - 1).mean()
print("party-swap asymmetry:", round(gap, 4))

2014 env -5.3 acc 0.9633


2016 env -0.6 acc 0.9677
2018 env +9.1 acc 0.9005
2020 env +3.5 acc 0.9654
2022 env -2.3 acc 0.9597
2024 env -2.6 acc 0.963
party-swap asymmetry: 0.0104


## 9 · The 2026 forecast  (HOML Ch. 3–4 *Estimating Probabilities*)
The environment comes from the final generic-ballot averages; one shared national polling error per simulated election.

In [10]:
gb = json.loads((B.RAW / "generic_ballot_2026.json").read_text())
avg = next(a for a in gb["rows"] if a["source"].lower().startswith("average"))
env26 = 100 * (avg["dem"] - avg["rep"]) / (avg["dem"] + avg["rep"])
d26 = B.load_districts(2026); X26 = B.features(d26, env26)
final = B.LogReg().fit(X_all[con], d.y[con])
contested = (d26.has_d == 1) & (d26.has_r == 1); only_d = (d26.has_d == 1) & (d26.has_r == 0)
rng = np.random.default_rng(42); seats = []
for s in rng.normal(0, 3.0, 2000):
    p = np.where(contested, final.predict_proba(X26.assign(env=env26 + s))[:,1], np.where(only_d, 1.0, 0.0))
    seats.append((rng.random(len(p)) < p).sum())
seats = np.array(seats)
print(f"env {env26:+.2f}  P(D majority) {np.mean(seats>=218):.2f}  median {np.median(seats):.0f}  80% {np.percentile(seats,10):.0f}-{np.percentile(seats,90):.0f}")

env +9.17  P(D majority) 0.99  median 242  80% 227-260


## 10 · Score it after Election Day  (HOML Ch. 2 *Launch, Monitor, and Maintain*)
The frozen forecast is `data/forecast/forecast_2026.csv`. After races are called:
```bash
.venv/bin/python ml/fetch_data.py --only 2026   # pull winners
.venv/bin/python ml/build.py                    # compares them with the frozen file
```

In [11]:
fz = pd.read_csv(B.OUT_FC / "forecast_2026.csv")
done = B.load_districts(2026).query("winner in ['D','R']").merge(fz, on="district")
print("called so far:", len(done))
if len(done):
    yy = (done.winner == "D").astype(int)
    print("accuracy", accuracy_score(yy, done.p_dem_logreg >= .5), "Brier", brier_score_loss(yy, done.p_dem_logreg))

called so far: 0
